# Assignment 3: The "Multimodal Sentiment Engine" Challenge

**Total Marks:** 20 | **Deadline:** 7:00 PM, 22nd March, 2026 | 
**Submission:** A zip file of the folder containing this notebook, and the csv/image files you will create.


---

## Setup

Run the cell below **once** to install all required packages and download NLTK data.

In [2]:
!pip install -r requirements.txt -q

import nltk
for pkg in ["wordnet", "averaged_perceptron_tagger_eng", "punkt_tab", "omw-1.4"]:
    nltk.download(pkg, quiet=True)
print("Setup complete!")

Setup complete!


In [3]:
import os, re, json, time, random, warnings
from collections import Counter
from itertools import combinations

from dotenv import load_dotenv
load_dotenv()

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag

warnings.filterwarnings("ignore")

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
LLM_MODEL = "google/gemini-2.0-flash-001"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Sentiment-bearing words to preserve during augmentation
PRESERVE_WORDS = {
    "amazing", "terrible", "awful", "excellent", "wonderful", "horrible",
    "great", "bad", "good", "worst", "best", "love", "hate", "boring",
    "fantastic", "brilliant", "pathetic", "outstanding", "dreadful",
    "superb", "mediocre",
}

print("Imports loaded. API key present:", bool(OPENROUTER_API_KEY))

Imports loaded. API key present: True


## Task 1: Data Consolidation & Classical Augmentation (5 Marks)

**Steps:**
1. Load all three CSVs and merge them
2. Train a baseline Logistic Regression on `gold_standard_100.csv` (TF-IDF features)
3. Filter `llm_labels_150.csv` -- keep only reviews where baseline confidence ≥ 0.65 AND agrees with LLM label
4. Deduplicate by review text $\rightarrow$ save `consolidated_base.csv`
5. Identify minority class, apply 2 augmentation methods (Synonym Replacement, Back Translation)
6. Quality filter augmented samples (Jaccard similarity)
7. Save `augmented_classical.csv` and `class_distribution.png`

In [4]:
gold = pd.read_csv("gold_standard_100.csv")
weak = pd.read_csv("weak_labels_200.csv")
llm  = pd.read_csv("llm_labels_150.csv")
print(f"Gold: {len(gold)}, Weak: {len(weak)}, LLM: {len(llm)}")

def train_baseline_model(train_df, text_col="review", label_col="label"):
    """Returns (vectorizer, classifier) trained on the given dataframe."""
    vec = TfidfVectorizer(max_features=5000, stop_words="english")
    X = vec.fit_transform(train_df[text_col])
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X, train_df[label_col])
    return vec, clf

vec, clf = train_baseline_model(gold)

#  1c. Filter LLM labels by confidence 
# TODO: Predict on llm reviews, keep where confidence >= 0.65 AND prediction matches LLM label

X_llm = vec.transform(llm['review'])
preds = clf.predict(X_llm)
probs = clf.predict_proba(X_llm)
confidence = np.max(probs, axis=1)
condition = (confidence >= 0.65) & (preds == llm['label'])
filtered_llm = llm[condition]
print(f"Filtered LLM labels: {len(filtered_llm)} out of {len(llm)}")

#  1d. Merge & deduplicate 
# TODO: Combine gold + weak + filtered_llm, drop_duplicates on "review"
# Save as consolidated_base.csv
df = pd.merge(gold, weak, on=["review", "label"], how="outer").merge(filtered_llm, on=["review", "label"], how="outer")
df = df.drop_duplicates(subset=["review"])
df.to_csv("consolidated_base.csv", index=False)

#  1e. Class distribution analysis 
# TODO: Count per class, identify minority, plot and save class_distribution.png

class_counts = df['label'].value_counts()
minority_class = class_counts.idxmin()
print("Class distribution:\n", class_counts)
print("Minority class:", minority_class)
plt.figure(figsize=(6,4))

plt.bar(class_counts.index, class_counts.values, color=['blue', 'orange', 'red'])
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.savefig("class_distribution.png")
plt.close()

Gold: 100, Weak: 220, LLM: 150
Filtered LLM labels: 27 out of 150
Class distribution:
 label
Negative    151
Neutral     115
Positive     62
Name: count, dtype: int64
Minority class: Positive


In [5]:
#  1f. Augmentation functions 

def synonym_replacement(text, replace_frac=0.15):
    """Replace 15-20% of words with WordNet synonyms. Preserve sentiment-bearing words."""
    # TODO: Implement using nltk.corpus.wordnet, nltk.pos_tag, word_tokenize
    
    words = word_tokenize(text)
    tagged = pos_tag(words)
    
    pos_map = {'NN': 'n', 'NNS': 'n', 'NNP': 'n', 'NNPS': 'n',
               'VB': 'v', 'VBD': 'v', 'VBG': 'v', 'VBN': 'v', 'VBP': 'v', 'VBZ': 'v',
               'JJ': 'a', 'JJR': 'a', 'JJS': 'a',
               'RB': 'r', 'RBR': 'r', 'RBS': 'r'}
    
    candidates = []
    for i, (word, pos) in enumerate(tagged):
        if word.lower() in PRESERVE_WORDS:
            continue
        wn_pos = pos_map.get(pos[:2], None)
        if wn_pos:
            synsets = wordnet.synsets(word.lower(), pos=wn_pos)
            if synsets:
                synonyms = set()
                for syn in synsets[:3]:
                    for lemma in syn.lemmas():
                        if lemma.name().lower() != word.lower():
                            synonyms.add(lemma.name())
                if synonyms:
                    candidates.append((i, list(synonyms)))
    
    num_replace = max(1, int(replace_frac * len(words)))
    if len(candidates) < num_replace:
        num_replace = len(candidates)
    
    selected = random.sample(candidates, num_replace) if candidates else []
    
    new_words = words[:]
    for idx, syns in selected:
        new_word = random.choice(syns)
        # Preserve case
        if words[idx].isupper():
            new_word = new_word.upper()
        elif words[idx].istitle():
            new_word = new_word.capitalize()
        new_words[idx] = new_word
    
    return ' '.join(new_words)

def back_translate(text, src="en", mid="hi"):
    """Translate English $\rightarrow$ Hindi $\rightarrow$ English using deep-translator GoogleTranslator."""
    from deep_translator import GoogleTranslator
    # TODO: Implement with error handling and rate-limit sleep
    
    try:
        translated = GoogleTranslator(source='en', target='hi').translate(text=text)
        time.sleep(1)
        back_translated = GoogleTranslator(source='hi', target='en').translate(text=translated)
        return back_translated
    except Exception as e:
        print(f"Translation error: {e}")
        return text  # Fallback to original text on error
    
def quality_filter(original, augmented):
    """Return True if augmented text passes Jaccard similarity (0.30–0.95)."""
    # TODO: Implement Jaccard similarity check
    
    original_set = set(original.lower().split())
    augmented_set = set(augmented.lower().split())
    intersection = original_set.intersection(augmented_set)
    union = original_set.union(augmented_set)
    
    jaccard_sim = len(intersection) / len(union) if union else 0
    return 0.30 <= jaccard_sim <= 0.95

#  1g. Apply augmentation to minority class 
# TODO: For each minority-class sample, generate 2 augmented versions (one per method)
# TODO: Apply quality_filter, keep only passing samples
# TODO: Save augmented_classical.csv

added_rows = []
print("Starting augmentation...", end='\n\n')
for idx, row in df[df['label'] == minority_class].iterrows():
    original_text = row['review']
    
    aug_synonym = synonym_replacement(original_text)
    aug_backtrans = back_translate(original_text)
    
    if quality_filter(original_text, aug_synonym):
        added_rows.append({'review': aug_synonym, 'label': minority_class})
    if quality_filter(original_text, aug_backtrans):
        added_rows.append({'review': aug_backtrans, 'label': minority_class})
        
df = pd.concat([df, pd.DataFrame(added_rows)], ignore_index=True)
df.to_csv("augmented_classical.csv", index=False)


print("/nAugmentation complete. Total samples after augmentation:", len(df))

Starting augmentation...

/nAugmentation complete. Total samples after augmentation: 451


## Task 2: LLM-Based Synthetic Review Generation (5 Marks)

**Steps:**
1. Design a few-shot prompt with 3-4 gold-standard examples
2. Use OpenRouter API (via `openai` package) to generate 300 synthetic reviews in batches of 20
3. Calculate diversity metrics: Self-BLEU per class
4. Run sentiment consistency check with baseline model $\rightarrow$ flag mismatches
5. Save `llm_generated_300.csv`, `llm_generated_flagged.csv`, `prompt_template.txt`, `diversity_report.txt`

In [6]:
from openai import OpenAI

client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)

#  2a. Design your few-shot prompt 
# TODO: Build a prompt string with 3-4 example reviews from gold standard
# Instruct the LLM to output JSON: [{"review": "...", "sentiment": "Positive", "movie": "..."}]
PROMPT_TEMPLATE = """You are a movie review generator. ..."""

# Save prompt to file
with open("prompt_template.txt", "w", encoding="utf-8") as f:
    f.write(PROMPT_TEMPLATE)

#  2b. Generate reviews in batches 
# TODO: Loop to generate ~300 reviews in batches of 20
# Target distribution: ~150 Positive, ~100 Negative, ~50 Neutral
# Parse JSON response, handle errors


#  2c. Diversity metrics 
# TODO: Calculate Self-BLEU per sentiment class using nltk.translate.bleu_score


#  2d. Sentiment consistency check 
# TODO: Use baseline model (vec, clf) to predict sentiment of each generated review
# TODO: Flag mismatches, save llm_generated_flagged.csv


#  2e. Save outputs 
# TODO: Save llm_generated_300.csv and diversity_report.txt

## Task 3: Multilingual Sentiment Translation (4 Marks)

**Steps:**
1. Sample 100 reviews (40 Pos, 40 Neg, 20 Neu), prioritize shorter reviews
2. Translate English $\rightarrow$ Hindi using `deep-translator` (`GoogleTranslator`)
3. Back-translate Hindi $\rightarrow$ English, compute BLEU score (threshold ≥ 0.3)
4. Check sentiment preservation on back-translated text
5. Manually verify 5 random samples
6. Save `bilingual_reviews.csv` with `bleu_score` and `quality_flag` columns

In [7]:
from deep_translator import GoogleTranslator
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

#  3a. Strategic sampling 
# TODO: From consolidated_base, sample 100 reviews (40 Pos, 40 Neg, 20 Neu)
# Prioritize shorter reviews (sort by length, take shortest)


#  3b. Translation pipeline 
# TODO: Translate each review English $\rightarrow$ Hindi using GoogleTranslator(source='en', target='hi')
# Add time.sleep(0.5) between calls to avoid rate limits


#  3c. Back-translation & BLEU 
# TODO: Translate Hindi $\rightarrow$ English
# Compute sentence BLEU between original and back-translated
# quality_flag = "PASS" if BLEU >= 0.3, else "FAIL"


#  3d. Sentiment preservation check 
# TODO: Predict sentiment on back-translated text, compare with original label


#  3e. Manual verification 
# TODO: Print 5 random samples for inspection


#  3f. Save 
# TODO: Save bilingual_reviews.csv with columns:
# review, label, hindi, back_translated, bleu_score, quality_flag

## Task 4: Multimodal Audio Generation (4 Marks)

**Steps:**
1. Select 30 reviews (10 per class) of varying lengths
2. Use `gTTS` to generate audio (`tld="com"`), convert mp3$\rightarrow$wav via `librosa`+`soundfile`
3. Extract audio features with `librosa`: duration, spectral centroid, zero crossing rate, MFCCs
4. Use `openai-whisper` (tiny model) to transcribe audio back to text, compute WER
5. Save `audio_samples/` folder, `audio_features.csv`, `audio_validation.csv`

In [18]:
from gtts import gTTS
import librosa, soundfile as sf

#  4a. Select 30 reviews (10 per class)
# TODO: Sample from consolidated_base, mix short and long reviews.
sampled_parts = []
per_class_target = 10

rng = np.random.default_rng()
sentiment_order = list(df['label'].dropna().unique())
rng.shuffle(sentiment_order)

for sentiment in sentiment_order:
    class_samples = df[df['label'] == sentiment].copy()
    class_samples = class_samples.dropna(subset=['review']).drop_duplicates(subset=['review'])
    class_samples['review_len'] = class_samples['review'].str.len()

    short_pool = class_samples[class_samples['review_len'] <= 100]
    long_pool = class_samples[class_samples['review_len'] > 100]

    short_n = min(5, len(short_pool))
    long_n = min(5, len(long_pool))

    short_samples = short_pool.sample(short_n) if short_n else short_pool.head(0)
    long_samples = long_pool.sample(long_n) if long_n else long_pool.head(0)

    selected = pd.concat([short_samples, long_samples])

    if len(selected) < per_class_target:
        remaining = class_samples.drop(index=selected.index, errors='ignore')
        fill_n = min(per_class_target - len(selected), len(remaining))
        if fill_n:
            fill_samples = remaining.sample(fill_n)
            selected = pd.concat([selected, fill_samples])

    selected = selected.sample(frac=1).head(per_class_target) if len(selected) else selected
    print(f"Selected {len(selected)} samples for sentiment {sentiment}")
    sampled_parts.append(selected[['review', 'label']])

sampled = pd.concat(sampled_parts, ignore_index=True)
sampled = sampled.drop_duplicates(subset=['review']).reset_index(drop=True)

if len(sampled) > 30:
    sampled = sampled.groupby('label', group_keys=False).apply(lambda x: x.sample(min(10, len(x)))).reset_index(drop=True)

os.makedirs("audio_samples", exist_ok=True)
sampled.to_csv("audio_samples/sampled_reviews.csv", index=False)


#  4b. TTS generation
# TODO: For each review, generate audio with gTTS (tld="com").
# TODO: Save as mp3, then load with librosa and re-save as .wav via soundfile.

for directory in ["mp3", "wav"]:
    dir_path = os.path.join("audio_samples", directory)
    if os.path.exists(dir_path):
        for file in os.listdir(dir_path):
            file_path = os.path.join(dir_path, file)
            if os.path.isfile(file_path):
                os.remove(file_path)
    else:
        os.makedirs(dir_path, exist_ok=True)

for idx, row in sampled.iterrows():
    review_text = str(row['review'])
    sentiment = str(row['label'])
    mp3filename_base = f"audio_samples/mp3/review_{idx}_{sentiment}"
    wavfilename_base = f"audio_samples/wav/review_{idx}_{sentiment}"

    tts = gTTS(text=review_text, lang='en', tld='com')
    mp3_path = f"{mp3filename_base}.mp3"
    wav_path = f"{wavfilename_base}.wav"

    tts.save(mp3_path)
    audio, sr = librosa.load(mp3_path, sr=16000)
    sf.write(wav_path, audio, sr)

print("Audio generation complete:", len(sampled), "samples")

Selected 10 samples for sentiment Positive
Selected 10 samples for sentiment Neutral
Selected 10 samples for sentiment Negative
Audio generation complete: 30 samples


In [22]:

#  4c. Audio feature extraction 
# TODO: For each wav, extract with librosa:
#   - duration (librosa.get_duration)
#   - spectral centroid (librosa.feature.spectral_centroid)
#   - zero crossing rate (librosa.feature.zero_crossing_rate)
#   - MFCCs (librosa.feature.mfcc, n_mfcc=13, take mean)
# Save audio_features.csv

audio_rows = []

for wav_file in sorted(os.listdir("audio_samples/wav")):
    if not wav_file.lower().endswith(".wav"):
        continue
    
    wav_path = os.path.join("audio_samples/wav", wav_file)
    filename = os.path.basename(wav_path)

    stem = os.path.splitext(filename)[0]
    parts = stem.split("_", 2)
    sentiment = parts[2] if len(parts) == 3 else "Unknown"

    try:
        audio, sr = librosa.load(wav_path, sr=16000)
        duration = librosa.get_duration(y=audio, sr=sr)
        spectral_centroid = float(np.mean(librosa.feature.spectral_centroid(y=audio, sr=sr)))
        zero_crossing_rate = float(np.mean(librosa.feature.zero_crossing_rate(y=audio)))
        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
        mfcc_mean = float(np.mean(mfccs))

        audio_rows.append({
            "filename": filename,
            "sentiment": sentiment,
            "duration": duration,
            "spectral_centroid": spectral_centroid,
            "zero_crossing_rate": zero_crossing_rate,
            "mfcc_mean": mfcc_mean,
        })
    except Exception as e:
        print(f"Skipping {filename}: {e}")

audio_data = pd.DataFrame(audio_rows, columns=[
    "filename", "sentiment", "duration", "spectral_centroid", "zero_crossing_rate", "mfcc_mean"
])
audio_data.to_csv("audio_features.csv", index=False)
print("audio_features.csv rows:", len(audio_data))
audio_data.head()

audio_features.csv rows: 30


,filename,sentiment,duration,spectral_centroid,zero_crossing_rate,mfcc_mean
0,review_0_Positive.wav,Positive,6.936,1608.128276,0.112894,-21.414810
1,review_10_Neutral.wav,Neutral,8.808,1902.272575,0.155247,-23.195898
2,review_11_Neutral.wav,Neutral,7.680,1776.540493,0.119335,-23.813313
3,review_12_Neutral.wav,Neutral,10.704,1751.775954,0.140762,-21.873108
4,review_13_Neutral.wav,Neutral,10.032,1644.973362,0.130052,-21.614147


In [27]:
#  4d. Whisper round-trip validation
# TODO: Transcribe each wav with Whisper.
# TODO: Compute WER (word-level Levenshtein distance / reference word count).
# TODO: Flag samples with WER > 0.25 and save audio_validation.csv.
import whisper

_whisper_model = whisper.load_model("tiny")

def _word_levenshtein_distance(ref_words, hyp_words):
    """Compute Levenshtein edit distance between two word lists."""
    m, n = len(ref_words), len(hyp_words)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if ref_words[i - 1] == hyp_words[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost,
            )
    return dp[m][n]

def compute_wer(reference, hypothesis):
    ref_words = re.findall(r"\w+", str(reference).lower())
    hyp_words = re.findall(r"\w+", str(hypothesis).lower())
    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else 1.0
    dist = _word_levenshtein_distance(ref_words, hyp_words)
    return dist / len(ref_words)

if 'sampled' in globals() and {'review', 'label'}.issubset(sampled.columns):
    sampled_for_lookup = sampled[['review', 'label']].reset_index(drop=True)
elif os.path.exists("audio_samples/sampled_reviews.csv"):
    sampled_for_lookup = pd.read_csv("audio_samples/sampled_reviews.csv")[['review', 'label']].reset_index(drop=True)
else:
    raise FileNotFoundError(
        "Missing sampled metadata. Run Task 4 cell 12 first to generate audio_samples/sampled_reviews.csv."
    )

sampled_lookup = sampled_for_lookup.to_dict("index")
validation_rows = []

for wav_file in sorted(os.listdir("audio_samples/wav")):
    if not wav_file.lower().endswith(".wav"):
        continue

    wav_path = os.path.join("audio_samples/wav", wav_file)

    base = os.path.splitext(wav_file)[0]
    parts = base.split("_", 2)
    if len(parts) < 3 or parts[0] != "review":
        continue

    try:
        sample_idx = int(parts[1])
    except ValueError:
        continue

    if sample_idx not in sampled_lookup:
        continue

    reference_text = sampled_lookup[sample_idx]["review"]
    true_label = sampled_lookup[sample_idx]["label"]

    try:
        audio_wave, sr = librosa.load(wav_path, sr=16000)
        result = _whisper_model.transcribe(audio_wave, fp16=False)
        transcript = result.get("text", "").strip()
        wer = compute_wer(reference_text, transcript)
        quality_flag = "FAIL" if wer > 0.25 else "PASS"
    except Exception as e:
        transcript = ""
        wer = np.nan
        quality_flag = f"ERROR: {type(e).__name__}"
        print(f"Transcription failed for {wav_file}: {e}")

    validation_rows.append({
        "filename": wav_file,
        "label": true_label,
        "reference": reference_text,
        "transcript": transcript,
        "wer": round(wer, 4),
        "quality_flag": quality_flag,
    })

audio_validation = pd.DataFrame(validation_rows)
audio_validation.to_csv("audio_validation.csv", index=False)

print("Whisper validation complete.")
print("Total files processed:", len(audio_validation))
if len(audio_validation) and "wer" in audio_validation.columns:
    print("High-WER files (>0.25):", int((audio_validation["wer"] > 0.25).sum()))
audio_validation.head()

Whisper validation complete.
Total files processed: 30
High-WER files (>0.25): 3


,filename,label,reference,transcript,wer,quality_flag
0,review_0_Positive.wav,Positive,It managed to be breathtaking without trying t...,It managed to be breathtaking without trying t...,0.0,PASS
1,review_10_Neutral.wav,Neutral,"It was... fine. It has some good moments, espe...","It was fine. It has some good moments, especia...",0.0,PASS
2,review_11_Neutral.wav,Neutral,"Ideally, this should have been great, but some...","Ideally, this should have been great, but some...",0.0,PASS
3,review_12_Neutral.wav,Neutral,I walked in with low expectations and they wer...,I walked in with low expectations and they wer...,0.0,PASS
4,review_13_Neutral.wav,Neutral,I have mixed feelings about this one. While th...,I have mixed feelings about this one. While th...,0.0,PASS


In [28]:
audio_validation[audio_validation["wer"] > 0.25]

,filename,label,reference,transcript,wer,quality_flag
21,review_29_Negative.wav,Negative,It serves its purpose as popcorn filler. It do...,It serves its purpose as Bob Confiler. It does...,0.2778,FAIL
25,review_5_Positive.wav,Positive,"Wow , just wow . Every scene matte earned , an...","Well, just were, every scene maturned, and the...",0.4118,FAIL
28,review_8_Positive.wav,Positive,This motion-picture_show be a triumph in every...,This motion picture under school show be a tri...,0.2727,FAIL


## Task 5: Final Dataset Assembly & Model Evaluation (2 Marks)

**Steps:**
1. Merge all datasets: `consolidated_base.csv` + `augmented_classical.csv` + `llm_generated_300.csv` (excluding flagged) + English text from `bilingual_reviews.csv`
2. Deduplicate $\rightarrow$ save `final_augmented_dataset.csv`
3. Use `BlackBoxEvaluator` from `evaluator.py` to compare baseline vs augmented accuracy

In [ ]:
from evaluator import BlackBoxEvaluator

#  5a. Assemble final dataset 
# TODO: Load consolidated_base, augmented_classical, llm_generated_300, bilingual_reviews
# Exclude flagged LLM reviews
# Merge all, deduplicate on "review" column
# Save final_augmented_dataset.csv


#  5b. Black-Box Evaluation 
evaluator = BlackBoxEvaluator()
test_df = pd.read_csv("gold_standard_100.csv")

# Baseline evaluation (small dataset only)
# TODO: baseline_acc = evaluator.run_evaluation(consolidated_base, test_df)

# Augmented evaluation (full dataset)
# TODO: augmented_acc = evaluator.run_evaluation(final_augmented, test_df)

# Print comparison
# print(f"Baseline accuracy:  {baseline_acc:.2%}")
# print(f"Augmented accuracy: {augmented_acc:.2%}")
# print(f"Improvement:        {augmented_acc - baseline_acc:+.2%}")